# Unit 4: Hidden Markov Models using AI4Bharat LAHAJA

**Project objective:** Implement a Hidden Markov Model (HMM) on Hindi speech audio and explore whether acoustic sequence likelihoods vary across speakers' native states.

This notebook uses all available 6,152 LAHAJA recordings. Audio is read from the `audio_filepath` field embedded in the four Parquet shards. MFCCs are kept as short ordered sequences for the HMM rather than being averaged into one vector.

**Important interpretation:** `native_state` is geographic metadata, not a verified Hindi dialect label. Results are exploratory and should not be described as definitive dialect classification.

## Cell 1 — Install and import libraries

This installs `hmmlearn` for Gaussian HMMs and imports the libraries for loading Parquet data, processing audio, plotting, and evaluating HMM sequences. Run this cell first.

In [ ]:
%pip -q install hmmlearn librosa soundfile pyarrow

## Cell 2 — Import libraries and set parameters

The notebook uses 13 MFCC coefficients per audio frame. To keep computation manageable across all 6,152 recordings, each recording is represented by at most 60 evenly spaced frames. The frame order is retained, so the HMM still models sequential acoustic patterns.

In [ ]:
import os
import io
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from hmmlearn.hmm import GaussianHMM
from scipy.special import logsumexp
from google.colab import drive

warnings.filterwarnings("ignore")
np.random.seed(42)

N_MFCC = 13
MAX_FRAMES = 60
N_STATES = 4
HMM_ITERATIONS = 10
SAMPLE_RATE = 16000

## Cell 3 — Connect Google Drive and find the four Parquet shards

Mount Google Drive and search for the four LAHAJA test shards used in your earlier dataset analysis. If any file is not found, check that all four files are stored somewhere in your Drive.

In [ ]:
drive.mount("/content/drive")

required_files = [
    "test-00000-of-00004.parquet",
    "test-00001-of-00004.parquet",
    "test-00002-of-00004.parquet",
    "test-00003-of-00004.parquet"
]

found_files = {}

for root, dirs, files in os.walk("/content/drive"):
    for filename in files:
        if filename in required_files:
            found_files[filename] = os.path.join(root, filename)

for filename in required_files:
    print(filename, "->", found_files.get(filename, "NOT FOUND"))

if len(found_files) != 4:
    raise FileNotFoundError("Could not find all four LAHAJA Parquet shards in Google Drive.")

## Cell 4 — Load metadata and check the dataset

The metadata is loaded without audio bytes first, which keeps this step lighter. The notebook checks the expected sample count and displays the columns used later for regional analysis.

In [ ]:
metadata_columns = [
    "duration", "scenario", "fname", "native_language", "gender",
    "age_group", "native_state", "native_district", "sp_id", "lang"
]

metadata_parts = []

for filename in required_files:
    part = pd.read_parquet(found_files[filename], columns=metadata_columns)
    part["source_shard"] = filename
    metadata_parts.append(part)

metadata = pd.concat(metadata_parts, ignore_index=True)

print("Metadata shape:", metadata.shape)
print("Unique language tags:", metadata["lang"].dropna().unique())
print("Unique speakers:", metadata["sp_id"].nunique())
display(metadata.head())
display(metadata[["duration", "native_state", "scenario"]].describe(include="all").T)

## Cell 5 — Explore the regional sample distribution

This plot shows how many recordings come from each native state. This matters because state groups are imbalanced, so later regional averages should be interpreted alongside the number of recordings in each group.

In [ ]:
state_counts = metadata["native_state"].value_counts().sort_values()

plt.figure(figsize=(10, 7))
state_counts.plot(kind="barh")
plt.xlabel("Number of recordings")
plt.ylabel("Native state")
plt.title("LAHAJA recordings by native state")
plt.tight_layout()
plt.show()

print("Total recordings in metadata:", len(metadata))
print("Total duration in hours:", round(metadata["duration"].sum() / 3600, 2))

## Cell 6 — Define the audio sequence extractor

Each embedded audio object contains audio bytes. The function loads the audio at 16 kHz, extracts 13 MFCCs over time, and downsamples the sequence to at most 60 frames. It returns a matrix with shape `(frames, 13)`. Unlike mean MFCC features, this preserves the order of frames for HMM analysis.

In [ ]:
def extract_mfcc_sequence(audio_object):
    if audio_object is None:
        raise ValueError("Audio object is missing")

    if hasattr(audio_object, "as_py"):
        audio_object = audio_object.as_py()

    audio_bytes = audio_object.get("bytes")
    if audio_bytes is None:
        raise ValueError("Audio bytes are missing")

    y, sr = librosa.load(io.BytesIO(audio_bytes), sr=SAMPLE_RATE, mono=True)

    if len(y) == 0:
        raise ValueError("Audio is empty")

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC,
        n_fft=1024,
        hop_length=512
    ).T

    if len(mfcc) > MAX_FRAMES:
        selected = np.linspace(0, len(mfcc) - 1, MAX_FRAMES).astype(int)
        mfcc = mfcc[selected]

    return mfcc.astype(np.float32)

## Cell 7 — Extract ordered MFCC sequences from all recordings

This is the main audio-processing step. It reads one shard at a time and processes every audio object, so the notebook attempts all 6,152 recordings without loading every audio byte into memory at once. Failed recordings are recorded in a separate list; they are not silently counted as successful.

In [ ]:
sequences = []
sequence_rows = []
failed_rows = []
processed = 0
start_time = time.time()

for filename in required_files:
    parquet_file = pq.ParquetFile(found_files[filename])
    metadata_part = pd.read_parquet(found_files[filename], columns=metadata_columns)

    row_index = 0
    for batch in parquet_file.iter_batches(batch_size=1, columns=["audio_filepath"]):
        audio_object = batch.column(0)[0].as_py()
        row = metadata_part.iloc[row_index].to_dict()

        try:
            sequence = extract_mfcc_sequence(audio_object)
            sequences.append(sequence)
            sequence_rows.append(row)
        except Exception as error:
            failed_rows.append({
                "source_shard": filename,
                "row_index": row_index,
                "error": str(error)
            })

        row_index += 1
        processed += 1

        if processed % 250 == 0:
            print(f"Processed {processed} recordings")

sequence_metadata = pd.DataFrame(sequence_rows).reset_index(drop=True)
failed_df = pd.DataFrame(failed_rows)

print("Attempted recordings:", processed)
print("Successful sequences:", len(sequences))
print("Failed recordings:", len(failed_df))
print("Elapsed minutes:", round((time.time() - start_time) / 60, 2))
print("First sequence shape:", sequences[0].shape if sequences else "No sequences")
display(failed_df.head())

## Cell 8 — Inspect one MFCC sequence

The heatmap visualizes how the 13 MFCC coefficients change across frames in one recording. The horizontal axis is time in frame order; the vertical axis is the MFCC coefficient index.

In [ ]:
sample_sequence = sequences[0]

plt.figure(figsize=(12, 4))
plt.imshow(sample_sequence.T, aspect="auto", origin="lower")
plt.xlabel("Frame index")
plt.ylabel("MFCC coefficient")
plt.title("MFCC sequence from one LAHAJA recording")
plt.colorbar(label="MFCC value")
plt.tight_layout()
plt.show()

print("Frames:", sample_sequence.shape[0])
print("MFCC dimensions:", sample_sequence.shape[1])

## Cell 9 — Prepare sequences for the HMM

`hmmlearn` accepts a single feature matrix plus a `lengths` list indicating where each recording starts and ends. The sequences are concatenated only for model fitting; the lengths preserve recording boundaries so transitions are not learned across two different speakers' recordings.

In [ ]:
valid_indices = [i for i, seq in enumerate(sequences) if len(seq) >= 2]
valid_sequences = [sequences[i] for i in valid_indices]
lengths = [len(seq) for seq in valid_sequences]
X_hmm = np.concatenate(valid_sequences, axis=0)

print("Number of recordings used:", len(valid_sequences))
print("Combined frame matrix:", X_hmm.shape)
print("Total sequence lengths:", sum(lengths))
print("Sequence count matches lengths:", len(lengths) == len(valid_sequences))

## Cell 10 — Train a Gaussian Hidden Markov Model

The HMM has four hidden states and diagonal Gaussian emissions. Hidden states are learned from the MFCC sequences; they are not pre-labelled phonemes, words, or dialects. The model is trained on all successfully extracted recordings.

In [ ]:
hmm_model = GaussianHMM(
    n_components=N_STATES,
    covariance_type="diag",
    n_iter=HMM_ITERATIONS,
    random_state=42,
    verbose=True,
    min_covar=1e-3
)

hmm_model.fit(X_hmm, lengths)

print("HMM training completed")
print("Converged:", hmm_model.monitor_.converged)
print("Iterations:", hmm_model.monitor_.iter)
print("Start probabilities:", hmm_model.startprob_)
print("Transition matrix:")
display(pd.DataFrame(hmm_model.transmat_))

## Cell 11 — Calculate sequence likelihoods for each recording

The HMM log likelihood indicates how well the trained model explains a recording's MFCC sequence. Dividing by the number of frames gives a length-normalized score that is easier to compare across recordings of different lengths. It is not a direct measure of dialect correctness.

In [ ]:
log_likelihoods = []
mean_log_likelihoods = []
frame_counts = []

for sequence in tqdm(valid_sequences):
    score = hmm_model.score(sequence)
    log_likelihoods.append(score)
    mean_log_likelihoods.append(score / len(sequence))
    frame_counts.append(len(sequence))

results = sequence_metadata.iloc[valid_indices].reset_index(drop=True).copy()
results["frame_count"] = frame_counts
results["hmm_log_likelihood"] = log_likelihoods
results["mean_log_likelihood"] = mean_log_likelihoods

print("Scored recordings:", len(results))
display(results[["native_state", "duration", "frame_count", "mean_log_likelihood"]].head())
display(results["mean_log_likelihood"].describe().to_frame("Mean log likelihood per frame"))

## Cell 12 — Demonstrate the Forward algorithm (sum-product)

The Forward algorithm sums the probabilities of all possible hidden-state paths that could produce the observed sequence. This implementation works in log space to reduce numerical underflow. Its final result should match `hmm_model.score()` for the same sequence, apart from small floating-point differences.

In [ ]:
def forward_log_likelihood(model, sequence):
    log_emissions = model._compute_log_likelihood(sequence)
    log_start = np.log(model.startprob_ + 1e-300)
    log_transition = np.log(model.transmat_ + 1e-300)

    alpha = np.zeros_like(log_emissions)
    alpha[0] = log_start + log_emissions[0]

    for t in range(1, len(sequence)):
        alpha[t] = log_emissions[t] + logsumexp(
            alpha[t - 1][:, None] + log_transition,
            axis=0
        )

    return logsumexp(alpha[-1])

sample_score = hmm_model.score(sample_sequence)
forward_score = forward_log_likelihood(hmm_model, sample_sequence)

print("HMM score:", sample_score)
print("Forward algorithm score:", forward_score)
print("Absolute difference:", abs(sample_score - forward_score))

## Cell 13 — Demonstrate the Backward algorithm

The Backward algorithm works from the end of the sequence toward the beginning. Combining the first forward values with the backward values gives the same total sequence likelihood. This cell uses the same learned HMM parameters and the same sample sequence.

In [ ]:
def backward_log_likelihood(model, sequence):
    log_emissions = model._compute_log_likelihood(sequence)
    log_transition = np.log(model.transmat_ + 1e-300)
    log_start = np.log(model.startprob_ + 1e-300)

    beta = np.zeros_like(log_emissions)

    for t in range(len(sequence) - 2, -1, -1):
        beta[t] = logsumexp(
            log_transition + log_emissions[t + 1][None, :] + beta[t + 1][None, :],
            axis=1
        )

    return logsumexp(log_start + log_emissions[0] + beta[0])

backward_score = backward_log_likelihood(hmm_model, sample_sequence)

print("HMM score:", sample_score)
print("Backward algorithm score:", backward_score)
print("Absolute difference:", abs(sample_score - backward_score))

## Cell 14 — Demonstrate the Viterbi algorithm

Viterbi finds the single most likely sequence of hidden states for the selected recording. The plot shows the inferred state at each MFCC frame. These state IDs are arbitrary learned acoustic states and should not be named as phonemes without further validation.

In [ ]:
viterbi_states = hmm_model.predict(sample_sequence)

plt.figure(figsize=(12, 3))
plt.step(np.arange(len(viterbi_states)), viterbi_states, where="mid")
plt.xlabel("Frame index")
plt.ylabel("Hidden state")
plt.title("Viterbi hidden-state path for one recording")
plt.yticks(range(N_STATES))
plt.tight_layout()
plt.show()

print("Predicted hidden states:", np.unique(viterbi_states))
print("State path length:", len(viterbi_states))

## Cell 15 — Compare HMM likelihoods across native states

The analysis keeps states with at least 50 successfully processed recordings to reduce the instability of very small groups. It reports both the number of recordings and the mean normalized log likelihood. Differences may reflect recording conditions, speaker composition, duration, or other factors—not only regional speech differences.

In [ ]:
state_summary = (
    results.groupby("native_state")
    .agg(
        recordings=("mean_log_likelihood", "size"),
        mean_score=("mean_log_likelihood", "mean"),
        median_score=("mean_log_likelihood", "median"),
        std_score=("mean_log_likelihood", "std")
    )
    .query("recordings >= 50")
    .sort_values("mean_score")
)

display(state_summary)

plt.figure(figsize=(10, 7))
state_summary["mean_score"].plot(kind="barh", xerr=state_summary["std_score"])
plt.xlabel("Mean log likelihood per frame")
plt.ylabel("Native state")
plt.title("HMM acoustic sequence fit by native state")
plt.tight_layout()
plt.show()

## Cell 16 — Visualize the score distributions

A box plot shows the spread of normalized likelihoods in each state group. This helps avoid relying only on group averages. Groups may overlap substantially, and the plot alone does not establish a statistically significant regional difference.

In [ ]:
states_to_plot = state_summary.index.tolist()
plot_data = results[results["native_state"].isin(states_to_plot)].copy()

plt.figure(figsize=(13, 6))
plot_data.boxplot(column="mean_log_likelihood", by="native_state", rot=75, grid=False)
plt.xlabel("Native state")
plt.ylabel("Mean log likelihood per frame")
plt.title("Distribution of HMM likelihood by native state")
plt.suptitle("")
plt.tight_layout()
plt.show()

## Cell 17 — Check correlations with recording duration and scenario

This check helps identify possible confounding factors. A regional difference in HMM likelihood may be associated with recording duration or speaking scenario, so these variables should be considered before drawing conclusions.

In [ ]:
results["duration"] = pd.to_numeric(results["duration"], errors="coerce")

print("Correlation between duration and mean HMM likelihood:")
print(results[["duration", "mean_log_likelihood"]].corr().iloc[0, 1])

scenario_summary = (
    results.groupby("scenario")["mean_log_likelihood"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean")
)

display(scenario_summary)

plt.figure(figsize=(7, 4))
results.boxplot(column="mean_log_likelihood", by="scenario", grid=False)
plt.xlabel("Speech scenario")
plt.ylabel("Mean log likelihood per frame")
plt.title("HMM likelihood by speech scenario")
plt.suptitle("")
plt.tight_layout()
plt.show()

## Cell 18 — Save the results to Google Drive

This saves the per-recording scores, the regional summary, and any extraction failures as CSV files. These files can be used later for report tables and graphs.

In [ ]:
output_dir = "/content/drive/MyDrive/LAHAJA_HMM_Unit4"
os.makedirs(output_dir, exist_ok=True)

results.to_csv(os.path.join(output_dir, "hmm_recording_scores.csv"), index=False)
state_summary.to_csv(os.path.join(output_dir, "hmm_native_state_summary.csv"))
failed_df.to_csv(os.path.join(output_dir, "failed_audio_extractions.csv"), index=False)

print("Saved files to:", output_dir)
print(os.listdir(output_dir))

## Cell 19 — Final interpretation

Use the actual output values from your run to complete the conclusions. Do not invent results before the notebook has run.

- The Gaussian HMM models ordered MFCC frames using hidden acoustic states and transition probabilities.
- The Forward and Backward implementations should produce likelihoods close to the model's score.
- Viterbi produces the most likely hidden-state path for a recording.
- Native-state comparisons are exploratory. `native_state` is not an official dialect label, and speakers' native language is not necessarily the same as the dialect they use in a recording.
- The notebook attempts all recordings, but the number successfully processed is the number shown in Cell 7. Report failed samples and runtime if any extraction fails.